In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import make_pipeline

C:\Users\Hp Laptop\anaconda3\Lib\site-packages\sklearn\experimental\enable_hist_gradient_boosting.py:16: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [2]:
# Replace with your actual file path and target column name
data= pd.read_csv('C:/Users/Hp Laptop/Desktop/python/data/unassisted.csv')
birth=pd.DataFrame(data)

birth['unassist'].replace({'unassisted':1, 'assisted':0}, inplace=True)
# Define features and target

C:\Users\Hp Laptop\AppData\Local\Temp\ipykernel_14984\1097052638.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  birth['unassist'].replace({'unassisted':1, 'assisted':0}, inplace=True)
C:\Users\Hp Laptop\AppData\Local\Temp\ipykernel_14984\1097052638.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tru

In [3]:
X = birth.drop('unassist', axis=1)
y = birth['unassist']

In [4]:
X.shape

(186001, 21)

In [5]:
y.shape

(186001,)

In [6]:
y

0         1
1         1
2         1
3         1
4         1
         ..
185996    0
185997    0
185998    0
185999    0
186000    0
Name: unassist, Length: 186001, dtype: int64

In [7]:
X.info


<bound method DataFrame.info of         place interval_birth    last_child age_of_mathers wealth_status  \
0       rural             no   wanted then            >34          poor   
1       rural             no   wanted then            >34          poor   
2       rural             no   wanted then          15-19          poor   
3       rural             no   wanted then            >34          poor   
4       rural             no   wanted then          25-29          poor   
...       ...            ...           ...            ...           ...   
185996  urban             no  wanted later            >34          rich   
185997  urban             no  wanted later          25-29          rich   
185998  urban             no   wanted then          20-24          rich   
185999  urban             no  wanted later          30-34          rich   
186000  urban             no   wanted then          25-29          rich   

         education_maternal          hus_education maternaloccupati

In [8]:
# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [9]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [10]:
X_train.shape

(148800, 21)

In [11]:
X_test.shape

(37201, 21)

In [12]:
y_train.shape, y_test.shape


((148800,), (37201,))

In [13]:
# Define a column transformer for one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False), X.columns)  # Set sparse=False to get a dense array
    ])

# Apply one-hot encoding to training data
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Convert to dense array if necessary
X_train_encoded = np.array(X_train_encoded)
X_test_encoded = np.array(X_test_encoded)

# Apply SMOTE to balance the dataset using the encoded features
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_encoded, y_train)

In [14]:
# Define models with class weighting where applicable
models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(class_weight='balanced'),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'SVM': SVC(probability=True, class_weight='balanced'),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(),
    'CatBoost': CatBoostClassifier(verbose=0),
    'HistGradientBoosting': HistGradientBoostingClassifier(),
    'Neural Network': MLPClassifier(max_iter=1000, early_stopping=True)
}

In [ ]:
def evaluate_models(models, X_train_res, X_test_encoded, y_train_res, y_test):
    results = []
    plt.figure(figsize=(12, 8))
    
    for name, model in models.items():
        # Fit the model
        model.fit(X_train_res, y_train_res)
        
        # Make predictions
        predictions = model.predict(X_test_encoded)
        y_prob = model.predict_proba(X_test_encoded)[:, 1]  # Probability for ROC AUC
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions)
        f1 = f1_score(y_test, predictions)
        roc_auc = roc_auc_score(y_test, y_prob)
        
        # Store results
        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'F1 Score': f1,
            'AUC': roc_auc
        })
        
        # Compute ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')
    
    # Finalize ROC plot
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.show()
    
    # Create DataFrame for heatmap
    results_df = pd.DataFrame(results).set_index('Model')
    
    # Plot metrics heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(results_df, annot=True, cmap='viridis', fmt=".4f")
    plt.title('Model Evaluation Metrics for Imbalanced Dataset')
    plt.show()

# Call the function to evaluate models
evaluate_models(models, X_train_res, X_test_encoded, y_train_res, y_test)